# AMR Drug Repurposing — 01: Data Preprocessing

Fetches ESKAPE bioactivity data from ChEMBL, cleans and labels records, generates ECFP4 fingerprints, and saves all artefacts to `data/`.

## Paths

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent
# When running from notebooks/ dir, parent is the project root
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path.cwd()

DATA_DIR  = PROJECT_ROOT / 'data'
CKPT_DIR  = PROJECT_ROOT / 'checkpoints'
FIG_DIR   = PROJECT_ROOT / 'figures'

for d in [DATA_DIR, CKPT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"CKPT_DIR     : {CKPT_DIR}")
print(f"FIG_DIR      : {FIG_DIR}")

PROJECT_ROOT : /home/preonath/Desktop/Preonath/Jubayer/ChEMBL_Deeplearning/amr_drug_repurposing/notebooks
DATA_DIR     : /home/preonath/Desktop/Preonath/Jubayer/ChEMBL_Deeplearning/amr_drug_repurposing/notebooks/data
CKPT_DIR     : /home/preonath/Desktop/Preonath/Jubayer/ChEMBL_Deeplearning/amr_drug_repurposing/notebooks/checkpoints
FIG_DIR      : /home/preonath/Desktop/Preonath/Jubayer/ChEMBL_Deeplearning/amr_drug_repurposing/notebooks/figures


## Imports

In [1]:
import asyncio, httpx, json, time, warnings, os
import nest_asyncio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              classification_report, confusion_matrix,
                              roc_curve, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings('ignore')
nest_asyncio.apply()

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('mps' if torch.backends.mps.is_available()
                       else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

RuntimeError: Scikit-learn array API support was enabled but scipy's own support is not enabled. Please set the SCIPY_ARRAY_API=1 environment variable before importing sklearn or scipy. More details at: https://docs.scipy.org/doc/scipy/dev/api-dev/array_api.html

## ChEMBL async data fetcher

In [ ]:
BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

ESKAPE_ORGANISMS = [
    "Staphylococcus aureus",
    "Klebsiella pneumoniae",
    "Acinetobacter baumannii",
    "Pseudomonas aeruginosa",
    "Enterococcus faecium",
    "Enterobacter cloacae",
    "Escherichia coli",
    "Mycobacterium tuberculosis",
]

async def fetch_page(client, url, params, retries=3):
    for attempt in range(retries):
        try:
            r = await client.get(url, params=params, timeout=60)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == retries - 1:
                print(f"Failed after {retries} attempts: {e}")
                return None
            await asyncio.sleep(2 ** attempt)

async def fetch_organism_activities(client, organism, limit=1000):
    params = {
        "target_organism": organism,
        "standard_type__in": "MIC,IC50,MBC,Inhibition",
        "standard_value__isnull": False,
        "assay_type__in": "F,B",
        "limit": limit,
        "offset": 0,
        "format": "json",
        "fields": ("molecule_chembl_id,canonical_smiles,standard_type,"
                   "standard_value,standard_units,target_organism,"
                   "assay_chembl_id,activity_comment,pchembl_value"),
    }
    data = await fetch_page(client, f"{BASE_URL}/activity.json", params)
    if data is None:
        return []
    total = data["page_meta"]["total_count"]
    records = data["activities"]
    print(f"  {organism}: {total} total records")
    max_records = min(total, 20000)
    offsets = range(limit, max_records, limit)
    tasks = [fetch_page(client, f"{BASE_URL}/activity.json", {**params, "offset": o})
             for o in offsets]
    pages = await asyncio.gather(*tasks)
    for page in pages:
        if page:
            records.extend(page["activities"])
    return records

async def fetch_all_eskape(organisms):
    all_records = []
    limits = httpx.Limits(max_connections=5, max_keepalive_connections=5)
    async with httpx.AsyncClient(limits=limits) as client:
        for org in organisms:
            print(f"Fetching: {org}")
            records = await fetch_organism_activities(client, org)
            all_records.extend(records)
            await asyncio.sleep(1)
    return all_records

## Download & cache raw data

In [ ]:
RAW_PATH = DATA_DIR / 'eskape_raw.csv'

if RAW_PATH.exists() and RAW_PATH.stat().st_size > 0:
    print("Loading cached raw data...")
    df_raw = pd.read_csv(RAW_PATH)
    if df_raw.empty:
        df_raw = pd.DataFrame()
else:
    df_raw = pd.DataFrame()

if df_raw.empty:
    print("Fetching from ChEMBL API...")
    t0 = time.time()
    records = await fetch_all_eskape(ESKAPE_ORGANISMS)
    df_raw = pd.DataFrame(records)
    if not df_raw.empty:
        df_raw.to_csv(RAW_PATH, index=False)
        print(f"Done in {(time.time()-t0)/60:.1f} min")
    else:
        print("No records returned.")

print(f"Raw shape: {df_raw.shape}")
print(df_raw.head(3))

## Data cleaning & activity labelling

In [ ]:
def clean_and_label(df):
    df = df.copy()
    df = df.dropna(subset=['canonical_smiles', 'standard_value'])
    df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
    df = df.dropna(subset=['standard_value'])
    df = df[df['standard_value'] > 0]
    df = df[df['canonical_smiles'].str.len() > 5]
    df = df[~df['canonical_smiles'].str.contains(r'\*', regex=True)]

    def convert_to_uM(row):
        val   = row['standard_value']
        units = str(row.get('standard_units', '')).lower().strip()
        stype = str(row.get('standard_type', '')).upper()
        if stype == 'INHIBITION':
            return val
        if units in ['nm', 'nanomolar']:
            return val / 1000
        elif units in ['mm', 'millimolar']:
            return val * 1000
        return val  # µg/mL ≈ µM approximation common in AMR

    df['value_uM'] = df.apply(convert_to_uM, axis=1)

    def label(row):
        stype = str(row['standard_type']).upper()
        val   = row['value_uM']
        if stype == 'MIC':        return 1 if val <= 8  else 0
        elif stype == 'IC50':     return 1 if val <= 10 else 0
        elif stype == 'MBC':      return 1 if val <= 16 else 0
        elif stype == 'INHIBITION': return 1 if val >= 50 else 0
        return np.nan

    df['active'] = df.apply(label, axis=1).astype('float')
    df = df.dropna(subset=['active'])
    df['active'] = df['active'].astype(int)
    df = (df.sort_values('value_uM')
            .drop_duplicates(subset=['molecule_chembl_id', 'target_organism'], keep='first'))
    return df.reset_index(drop=True)

df_clean = clean_and_label(df_raw)
df_clean.to_csv(DATA_DIR / 'eskape_clean.csv', index=False)

print(f"Clean dataset: {df_clean.shape}")
vc = df_clean['active'].value_counts()
print(f"Active   (1): {vc.get(1,0):,}  ({vc.get(1,0)/len(df_clean)*100:.1f}%)")
print(f"Inactive (0): {vc.get(0,0):,}  ({vc.get(0,0)/len(df_clean)*100:.1f}%)")
print(df_clean.groupby('target_organism')['active']
              .agg(['sum','count'])
              .rename(columns={'sum':'active','count':'total'})
              .assign(pct=lambda x: (x.active/x.total*100).round(1))
              .sort_values('total', ascending=False))

## Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

org_counts = (df_clean.groupby(['target_organism', 'active'])
                       .size().unstack(fill_value=0))
org_counts.columns = ['Inactive', 'Active']
org_counts.plot(kind='barh', ax=axes[0], color=['#B5D4F4','#1D9E75'])
axes[0].set_title('Active vs Inactive per Organism')
axes[0].set_xlabel('Compound count')

pchem = df_clean['pchembl_value'].dropna().astype(float)
if len(pchem) > 100:
    axes[1].hist(pchem, bins=40, color='#7F77DD', edgecolor='white', alpha=0.8)
    axes[1].axvline(5, color='#D85A30', linestyle='--', label='pIC50=5 (10µM)')
    axes[1].set_title('pChEMBL Value Distribution')
    axes[1].set_xlabel('pChEMBL (higher = more potent)')
    axes[1].legend()
else:
    log_vals = np.log10(df_clean['value_uM'].clip(1e-3, 1e5))
    axes[1].hist(log_vals, bins=40, color='#7F77DD', edgecolor='white', alpha=0.8)
    axes[1].set_title('Log₁₀(Activity Value µM)')
    axes[1].set_xlabel('log₁₀(value)')

stype_counts = df_clean['standard_type'].value_counts().head(8)
axes[2].bar(stype_counts.index, stype_counts.values, color='#EF9F27')
axes[2].set_title('Assay Type Distribution')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eda.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIG_DIR / 'eda.png'}")

## Molecular fingerprint generation

In [ ]:
def smiles_to_ecfp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nbits)
    return np.array(fp)

print("Generating ECFP4 fingerprints (2048 bits)...")
fps, labels, smiles_valid = [], [], []

for _, row in tqdm(df_clean.iterrows(), total=len(df_clean)):
    fp = smiles_to_ecfp(row['canonical_smiles'])
    if fp is not None:
        fps.append(fp)
        labels.append(row['active'])
        smiles_valid.append(row['canonical_smiles'])

X = np.array(fps, dtype=np.float32)
y = np.array(labels, dtype=np.int32)

print(f"Fingerprint matrix : {X.shape}")
print(f"Active: {y.sum()} | Inactive: {(y==0).sum()}")

np.save(DATA_DIR / 'X_ecfp.npy',      X)
np.save(DATA_DIR / 'y_labels.npy',    y)
np.save(DATA_DIR / 'smiles_valid.npy', np.array(smiles_valid))
print("Saved fingerprints to data/.")